# Cleaning the catalogue (Catalog 2)

Catalog 2 twin of `01_cleaning.ipynb`. Turns the raw CHIME/FRB Catalog 2
table into a tidy, scaled feature table ready for outlier detection, using the
same steps as the Cat 1 cleaning so the two are directly comparable. Differences
from Cat 1 are called out in the cells. Every decision is written into the cells as
comments so nothing happens off-screen. Run top to bottom.

Decisions carried over from the 2026-06-05 session log:
- `sub_num`: KEEP every sub-component row (one row per sub-burst), mirroring Cat 1.
  Burst-level quantities (DM, flux, fluence) repeat across a burst's components;
  only morphology (width, scattering) varies. The per-source aggregation in the
  validation step is what stops multi-component bursts dominating.
- Quality flags: drop a row if ANY of `excluded_flag`, `sidelobe_flag`,
  `intrachan_flag` is set. This is the Cat 2 split of Cat 1's single `Flag`.

Decision made 2026-06-07:
- `scat_time` is DROPPED as a feature for Cat 2, leaving EIGHT features (Cat 1 used
  nine). Cat 2 records scattering as 0 for ~60% of bursts (non-detections, no error
  bar) rather than as the small positive upper limits Cat 1 used. Keeping it would let
  the outlier methods key on 'was scattering measurable at all', which tracks
  brightness/SNR -- a selection effect, not intrinsic morphology. Scattering is
  revisited properly in Phase 2 on the dynamic spectra, where the decay tail is
  visible directly instead of trusting a catalogue fit that gave up most of the time.

## Setup: load the shared reader and the raw catalogue

In [1]:
# Make the shared code in the project root's src/ importable. The reader lives in
# src/frb_anomaly/data.py so any notebook can reuse it (this is the 'module' habit).
import sys
from pathlib import Path

# Notebook lives at Phase 1/notebooks/, so two .parent hops reach the project root
# where src/, data/, etc live.
PROJECT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from frb_anomaly import data

# load_catalog2 returns just the DataFrame (a CSV carries no VOTable field metadata,
# so there is no `fields` list to unpack like there was for Cat 1).
df = data.load_catalog2()
print(f'Loaded {len(df)} sub-burst rows, {df.shape[1]} columns')
print(f"  {df['tns_name'].nunique()} distinct bursts (the rest are extra sub-components)")

Loaded 5045 sub-burst rows, 60 columns
  4539 distinct bursts (the rest are extra sub-components)


## Drop CHIME's flagged bursts

Cat 1 had a single `Flag`. Cat 2 splits the same idea across three quality flags.
We drop a row if ANY of them is set: `excluded_flag` (events CHIME itself excludes
from population studies, e.g. commissioning-period detections), `sidelobe_flag`
(detected in a far sidelobe, so flux/position are unreliable), and `intrachan_flag`.
Keeping them would give us 'anomalies' that are really data faults. The other flags
(`citizen_science_flag`, `catalog1_flag`) are metadata, not quality cuts, so we leave
them alone.

In [2]:
quality_flags = ['excluded_flag', 'sidelobe_flag', 'intrachan_flag']

# Report what each flag costs on its own, then drop the union.
for c in quality_flags:
    print(f'  {c:15s}: {(df[c] != 0).sum()} rows set')

before = len(df)
keep = (df[quality_flags] == 0).all(axis=1)
df = df[keep].copy()
print(f'Dropped {before - len(df)} flagged rows; {len(df)} remain')

  excluded_flag  : 409 rows set
  sidelobe_flag  : 37 rows set
  intrachan_flag : 58 rows set
Dropped 494 flagged rows; 4551 remain


## Label repeaters (do not drop them yet)

`repeater_name` holds the repeating source's name, or is blank (NaN) for a one-off
burst. This is the Cat 2 equivalent of Cat 1's `RpName == '-9999'` sentinel. We only
LABEL here: the sanity-check pass uses everyone with this label, and the real hunt
later keeps only the one-offs.

In [3]:
name = df['repeater_name'].astype('string').str.strip()
df['is_repeater'] = name.notna() & (name != '')
print(df['is_repeater'].value_counts().rename({False: 'one-off', True: 'repeater'}))
print(f"  {df.loc[df['is_repeater'], 'repeater_name'].nunique()} distinct repeater sources")

is_repeater
one-off     3396
repeater    1155
Name: count, dtype: Int64
  81 distinct repeater sources


## Choose the feature columns for the burst's shape

Eight features: Cat 1's nine minus `scat_time` (see the header note on why scattering
is dropped for Cat 2). These describe what the burst looks like. We deliberately leave
out sky position, arrival time, exposure and fit-quality, none of which are about the
burst's shape.

In [4]:
FEATURES = {
    'dm_fitb':    'dispersion measure',
    'width_fitb': 'width',
    'flux':       'peak flux (lower limit)',
    'fluence':    'fluence (lower limit)',
    'sp_idx':     'spectral index',
    'sp_run':     'spectral running',
    'peak_freq':  'peak frequency',
}
# scat_time intentionally omitted for Cat 2 (non-detections stored as 0; see header).

# Bandwidth is not a single column; build it from the high/low detection frequencies.
df['bandwidth'] = df['high_freq'] - df['low_freq']
FEATURES['bandwidth'] = 'detection bandwidth (high_freq - low_freq)'

feature_cols = list(FEATURES)
print(f'Using {len(feature_cols)} features:')
for c, desc in FEATURES.items():
    print(f'  {c:11s} {desc}')

Using 8 features:
  dm_fitb     dispersion measure
  width_fitb  width
  flux        peak flux (lower limit)
  fluence     fluence (lower limit)
  sp_idx      spectral index
  sp_run      spectral running
  peak_freq   peak frequency
  bandwidth   detection bandwidth (high_freq - low_freq)


## Example data

Worth seeing the spread first. As in Cat 1, DM is in the hundreds-to-thousands while
spectral running has extreme values. That huge range across columns is exactly why we
transform and rescale.

In [5]:
df[feature_cols].describe().round(3).T

,count,mean,std,min,25%,50%,75%,max
dm_fitb,4473.0,597.101,404.317,87.756,313.430,484.067,763.173,3966.732
width_fitb,4473.0,0.002,0.003,0.000,0.000,0.001,0.002,0.027
flux,4109.0,1.358,2.563,0.050,0.455,0.713,1.281,64.440
fluence,4109.0,7.136,17.939,0.109,2.004,3.548,6.842,792.864
sp_idx,4473.0,45.494,172.904,-22.514,2.414,12.454,40.809,5900.074
sp_run,4473.0,-120.942,1950.467,-128961.939,-102.628,-38.666,-7.835,26.107
peak_freq,4473.0,504.248,97.396,400.200,430.017,474.353,554.773,800.183
bandwidth,4473.0,241.004,118.798,2.173,135.327,215.088,383.423,400.000


## Data cleaning (limits & missing values)

Limits: flux & fluence are lower limits. We keep the values but stay aware they are
boundaries, not exact (noted for the report). Missing values: rather than invent
numbers to fill gaps, we keep only bursts that have a complete set of features, and
report how many we lose. Cat 2 has real gaps here (notably flux/fluence on some rows),
unlike Cat 1 which lost none.

In [6]:
# Keep tns_name + sub_num as the identity (tns_name alone repeats across sub-components),
# plus the repeater label and source name for the validation step. catalog1_flag is kept
# as useful metadata for the later instrument-vs-astrophysics triage.
id_cols = ['tns_name', 'sub_num', 'is_repeater', 'repeater_name', 'catalog1_flag']
feat = df[id_cols + feature_cols].copy()

complete = feat.dropna(subset=feature_cols)
lost = len(feat) - len(complete)
print(f'{len(complete)} of {len(feat)} rows have all {len(feature_cols)} features '
      f'({lost} dropped for missing values)')
for c in feature_cols:
    n = feat[c].isna().sum()
    if n:
        print(f'  {c:11s}: {n} missing')

4109 of 4551 rows have all 8 features (442 dropped for missing values)
  dm_fitb    : 78 missing
  width_fitb : 78 missing
  flux       : 442 missing
  fluence    : 442 missing
  sp_idx     : 78 missing
  sp_run     : 78 missing
  peak_freq  : 78 missing
  bandwidth  : 78 missing


## Transform and rescale

Same transforms as Cat 1. Several quantities span orders of magnitude, so log-scale
them first, then put everything on a common scale (mean 0, spread 1) so the outlier
maths treats each feature fairly instead of mostly reacting to the big-numbered columns.

In [7]:
# DM and width are strictly positive -> plain log spreads them out.
log_plain = ['dm_fitb', 'width_fitb']
# Flux and fluence can be ~0 -> log1p = log(1 + x) is safe at zero.
log_1p = ['flux', 'fluence']

# Safety check: plain log needs strictly positive inputs. (This is the guard that
# caught scat_time's zeros and led us to drop it; kept as a general check.)
for c in log_plain:
    nbad = (complete[c] <= 0).sum()
    if nbad:
        print(f'  WARNING: {c} has {nbad} values <= 0 that plain log cannot take')

transformed = complete.copy()
transformed[log_plain] = np.log(transformed[log_plain])
transformed[log_1p] = np.log1p(transformed[log_1p])

scaler = StandardScaler()
X = scaler.fit_transform(transformed[feature_cols])
scaled = pd.DataFrame(X, columns=feature_cols, index=transformed.index)

print('Each feature now has ~mean 0 and ~std 1:')
print(scaled.describe().loc[['mean', 'std']].round(2).T)

Each feature now has ~mean 0 and ~std 1:
            mean  std
dm_fitb     -0.0  1.0
width_fitb   0.0  1.0
flux        -0.0  1.0
fluence      0.0  1.0
sp_idx      -0.0  1.0
sp_run       0.0  1.0
peak_freq   -0.0  1.0
bandwidth    0.0  1.0


## Save the cleaned tables for the outlier step

Two files the next notebook will load, named with a `catalog2_` prefix so they sit
alongside the Cat 1 outputs without clobbering them:
- scaled features: what the outlier maths runs on
- raw features: original values, needed later to explain WHY a burst is an outlier

In [8]:
ids = complete[id_cols].reset_index(drop=True)
scaled_out = pd.concat([ids, scaled.reset_index(drop=True)], axis=1)
raw_out = pd.concat([ids, complete[feature_cols].reset_index(drop=True)], axis=1)

# Phase 1 outputs go into data/processed/phase_1/.
processed_dir = PROJECT / 'data' / 'processed' / 'phase_1'
processed_dir.mkdir(parents=True, exist_ok=True)
scaled_out.to_csv(processed_dir / 'catalog2_features_scaled.csv', index=False)
raw_out.to_csv(processed_dir / 'catalog2_features_raw.csv', index=False)

print('Saved:')
print('  data/processed/phase_1/catalog2_features_scaled.csv  (for the outlier maths)')
print('  data/processed/phase_1/catalog2_features_raw.csv     (original values, for interpreting outliers)')
scaled_out.head()

Saved:
  data/processed/phase_1/catalog2_features_scaled.csv  (for the outlier maths)
  data/processed/phase_1/catalog2_features_raw.csv     (original values, for interpreting outliers)


,tns_name,sub_num,is_repeater,repeater_name,catalog1_flag,dm_fitb,width_fitb,flux,fluence,sp_idx,sp_run,peak_freq,bandwidth
0,FRB20180904A,0,False,NaN,1,-0.503988,-0.655361,1.843839,0.371065,-0.174360,0.210322,0.224368,0.552022
1,FRB20180906A,0,False,NaN,1,-0.406266,-0.639031,0.564015,-0.436646,-0.246748,0.299849,3.045587,1.323792
2,FRB20180906B,0,False,NaN,1,2.980802,-1.876918,-0.788708,-0.933303,-0.232186,0.253453,-0.391763,0.627412
3,FRB20180907D,0,False,NaN,1,1.766788,0.513545,-0.101745,0.154488,-0.136332,0.235686,2.192078,0.478481
4,FRB20180907A,0,False,NaN,1,0.948153,-1.194168,-0.123952,-0.402862,-0.170535,0.083623,-0.571739,-0.862512
